# Mineral Dissolution Kinetics
> Andrew McGallian

pH-dependent dissolution of CaO and CaCO3 in soil water starting from equilibrium with high-CO2 soil gas.

## CO2 System Solver (shared functions)

In [ ]:
import matplotlib.pyplot as plt

K1      = 10**(-6.5)
K2      = 10**(-10.8)
K_henry = 10**(1.5) / 1000   # mol/L/atm
K_water = 10**(-14)

def _alpha0(h): return h**2   / (h**2 + K1*h + K1*K2)
def _alpha1(h): return K1*h   / (h**2 + K1*h + K1*K2)
def _alpha2(h): return K1*K2  / (h**2 + K1*h + K1*K2)
def _alk_from_dic(DIC_L, h): return DIC_L*(_alpha1(h) + 2*_alpha2(h)) + K_water/h - h

def solve_co2_system(alk_m3, DIC_m3, iterations=100):
    """Solve CO2 system given alk and DIC in mol/m³. Returns (CO2_m3, CO3_m3, pCO2_ppm, pH)."""
    alk_L, DIC_L = alk_m3/1000, DIC_m3/1000
    pH_lo, pH_hi = 1.0, 14.0
    for _ in range(iterations):
        pH_mid = (pH_lo + pH_hi) / 2
        h = 10**(-pH_mid)
        if _alk_from_dic(DIC_L, h) < alk_L: pH_lo = pH_mid
        else:                                pH_hi = pH_mid
    pH = (pH_lo + pH_hi) / 2
    h  = 10**(-pH)
    CO2_L = DIC_L * _alpha0(h)
    CO3_L = DIC_L * _alpha2(h)
    return CO2_L*1000, CO3_L*1000, (CO2_L/K_henry)*1e6, pH

def co2_from_ppm(pCO2_ppm):
    """Dissolved CO2 concentration (mol/m³) for a given atmospheric pCO2 (ppm)."""
    return K_henry * (pCO2_ppm/1e6) * 1000

CO2, CO3, pCO2, pH = solve_co2_system(2.0, 2.0)
print(f"pH={pH:.2f}, pCO2={pCO2:.1f} ppm  (expected: pH≈8.59, pCO2≈507 ppm)")

## Mineral Dissolution Model

In [ ]:
def mineral_dissolution_model(soil_pCO2_ppm, k_CaO, k_CaCO3, dt=0.1, duration=10):
    """
    Simulate pH-dependent mineral dissolution in soil water.

    Dissolution rates scale with [H+] vs pH 5:
        rate = k * 10^(5 - pH)   [mol/m³/yr]

    CaO:   +2 eq alk per mol, no DIC change
    CaCO3: +2 eq alk per mol, +1 mol DIC per mol

    soil_pCO2_ppm : soil gas pCO2 (ppm)
    k_CaO         : CaO rate constant at pH 5 (mol/m³/yr)
    k_CaCO3       : CaCO3 rate constant at pH 5 (mol/m³/yr)
    """
    DIC = co2_from_ppm(soil_pCO2_ppm)      # initial DIC = dissolved CO2
    alk = 0.0                               # no alkalinity initially

    # Guard alk=0: use tiny positive value so solver finds the acidic pH
    _, _, _, pH = solve_co2_system(max(alk, 1e-10), DIC)

    time_points     = [0.0]
    DIC_conc        = [DIC]
    alk_conc        = [alk]
    pH_list         = [pH]
    rate_CaO_list   = [0.0]
    rate_CaCO3_list = [0.0]

    t = 0.0
    for _ in range(int(duration/dt)):
        t += dt
        time_points.append(t)

        r_CaO   = k_CaO   * 10**(5 - pH)
        r_CaCO3 = k_CaCO3 * 10**(5 - pH)

        alk += (2*r_CaO + 2*r_CaCO3) * dt
        DIC += r_CaCO3 * dt

        _, _, _, pH = solve_co2_system(max(alk, 1e-10), DIC)

        DIC_conc.append(DIC)
        alk_conc.append(alk)
        pH_list.append(pH)
        rate_CaO_list.append(r_CaO)
        rate_CaCO3_list.append(r_CaCO3)

    return [time_points, DIC_conc, alk_conc, pH_list, rate_CaO_list, rate_CaCO3_list]

## Experiments

In [ ]:
SOIL_PCO2 = 4000   # ppm — ~10x atmospheric
K_CAO     = 1e-3   # mol/m³/yr at pH 5
K_CACO3   = 1e-2   # 10x faster

cao_only   = mineral_dissolution_model(SOIL_PCO2, k_CaO=K_CAO,  k_CaCO3=0)
caco3_only = mineral_dissolution_model(SOIL_PCO2, k_CaO=0,       k_CaCO3=K_CACO3)
both       = mineral_dissolution_model(SOIL_PCO2, k_CaO=K_CAO,  k_CaCO3=K_CACO3)

## Plots

In [ ]:
runs   = [cao_only, caco3_only, both]
labels = ['CaO only', 'CaCO3 only', 'Both']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for run, label in zip(runs, labels):
    t, dic, alk, pH, r_cao, r_caco3 = run
    axes[0,0].plot(t, pH,    label=label)
    axes[0,1].plot(t, alk,   label=label)
    axes[1,0].plot(t, dic,   label=label)
    axes[1,1].plot(t, r_cao, label=label)

axes[0,0].set(xlabel='Time (yr)', ylabel='pH',           title='Solution pH')
axes[0,1].set(xlabel='Time (yr)', ylabel='mol/m³',       title='Alkalinity')
axes[1,0].set(xlabel='Time (yr)', ylabel='DIC (mol/m³)', title='Total CO2 (DIC)')
axes[1,1].set(xlabel='Time (yr)', ylabel='mol/m³/yr',    title='CaO Dissolution Rate (= CO2 Uptake Rate)')

for ax in axes.flatten():
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Mineral Dissolution Kinetics — Soil pCO2 = 4000 ppm', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()